# 07 Characterize Pioneer 21 cast data for irregular sample intervals

## Import modules

In [1]:
from os import path
import urllib.request as request
import glob
import re
import ast
import cmocean.cm as cmo
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy import stats
from sklearn import preprocessing

In [2]:
# Define generalized functions
# Version 27 Aug 2025: Does not retain integer number index,
# but makes more interchangeable with local and remote data loading.
def load_lisst(lisst_path, header):
    # Load LISST data from CSV
    lisst_df = pd.read_csv(lisst_path, names=header)
    try: lisst_df.head(1) 
    except: print("No LISST data found.")
    # Create LISST time vector for Dataframe Index
    lisst_time = pd.to_datetime(
         lisst_df[
              ["year", "month", "day",
               "hour", "minute", "second"]
               ],
         yearfirst=True, utc=True
         )
    lisst_df.insert(0, "time", lisst_time.values)
    lisst_df.set_index("time", drop=True, inplace=True)
    # print(lisst_data.head(1))
    # Convert data frames to xarray for easy manipulation
    lisst_ds = xr.Dataset.from_dataframe(lisst_df)
    # Create 2D array for binned volume concentration
    volumecon2D = list([])
    bins = list([])
    for var in lisst_ds.variables:
        if re.search("volumecon[0-9]+", var):
                bins.append(var)
                volumecon2D.append(lisst_ds[var])
    lisst_ds = lisst_ds.drop_vars(bins)
    str2num = lambda x: int(x.replace("volumecon", ""))
    bins = [str2num(x) for x in bins]
    lisst_ds["volume_concentration_2D"] = xr.concat(
        volumecon2D, pd.Index(bins, name="bin")
        )
    lisst_ds["volume_concentration_2D"] = lisst_ds["volume_concentration_2D"].assign_attrs(units="$\mu$L/L")
    return lisst_ds

<>:35: SyntaxWarning: invalid escape sequence '\m'
<>:35: SyntaxWarning: invalid escape sequence '\m'
C:\Users\kylene.cooley\AppData\Local\Temp\ipykernel_53108\1435033189.py:35: SyntaxWarning: invalid escape sequence '\m'
  lisst_ds["volume_concentration_2D"] = lisst_ds["volume_concentration_2D"].assign_attrs(units="$\mu$L/L")


In [9]:
# Load LISST CSV column names from json containing column headers
headers = pd.read_json("./inst_headers/lisst_hdr.json", typ='series', orient='records')
csvhdr = headers.iloc[0]

## Load local oatmilk trial data

In [ ]:
# Define paths to data and find files matching the file name pattern
LISST_PATH = "C:/Users/kylene.cooley/Documents/prtsz_bench_test"
DATA_PATH = "oatmilk-*[0-9]percent.csv"

flist = glob.glob(DATA_PATH, root_dir=LISST_PATH, recursive=True)

In [ ]:
# Load data to workspace
lisst0017 = load_lisst(path.join(LISST_PATH, flist[0]), csvhdr)
lisst0366 = load_lisst(path.join(LISST_PATH, flist[1]), csvhdr)
lisst0776 = load_lisst(path.join(LISST_PATH, flist[2]), csvhdr)
lisst1240 = load_lisst(path.join(LISST_PATH, flist[3]), csvhdr)
lisst1776 = load_lisst(path.join(LISST_PATH, flist[4]), csvhdr)
lisst2409 = load_lisst(path.join(LISST_PATH, flist[5]), csvhdr)
lisst3185 = load_lisst(path.join(LISST_PATH, flist[6]), csvhdr)
lisst4185 = load_lisst(path.join(LISST_PATH, flist[7]), csvhdr)

## Load Pioneer 21 LISST cast data

In [3]:
# Base url for data on Raw Data Archive
RDA_URL = "https://rawdata.oceanobservatories.org/files/cruise_data/Pioneer-MAB/Pioneer-21_AR87_2025-03-28/LISST/"

# Scrape the LISST readme file on the RDA
# to get CSV file names for each cast
def scrape_lisst_list(readme_file):
    castfiles = {}
    filenext = False
    text, urlheaders =  request.urlretrieve(RDA_URL+readme_file)
    with open(text) as f:
        for x in f:
            if "CAST" in x:
                # print(x)
                cast = x.replace("    ", "")[:-1]
                filenext = True
                continue
            if (filenext is True)&(".CSV" in x):
                # print(x)
                file = x.replace("    ", "")[:-1]
                castfiles[cast] = file
                filenext = False
            else:
                continue
    # Sort dict values into list of tuples with
    # key, value pairs in castfiles:
    pairs = list(castfiles.items())
    return pairs

In [4]:
# load cast data and return summary statistics
def summarize_cast_lisst(lisst_casts):
    # initialize arrays for results
    avg_optical_transmission = np.array([])
    avg_total_volume = np.array([])
    # loop through list of LISST cast files
    loop = lisst_casts.copy()
    while len(loop)>0:
        key, value = loop.pop(0)
        lisst_path = RDA_URL + value
        lisst_ds = load_lisst(lisst_path, csvhdr)
        lisst_ds = lisst_ds.assign_attrs(cast=key)
        # Mask data where depth < 0
        depth_mask = lisst_ds.time[lisst_ds.depth>=0].values
        lisst_ds = lisst_ds.sel(time=depth_mask)
        avg_optical_transmission = np.append(avg_optical_transmission,
            lisst_ds.optical_transmission.mean().values
        )
        avg_total_volume = np.append(avg_total_volume,
            lisst_ds.total_volumecon.mean().values
        )
    return avg_optical_transmission, avg_total_volume

In [5]:
# Load list of key, value pairs for each cast
readme_file = "AR87_LISST_README.txt"
pairs = scrape_lisst_list(readme_file)

In [6]:
# filter the pairs for casts with irregular sample intervals
irregular_casts = [
    "AR87a_CAST003_L10",
	"AR87a_CAST004_GL564_GL376",
	"AR87a_CAST006_SE",
	"AR87a_CAST007_SE",
	"AR87a_CAST012_AC1",
	"AR87a_CAST014_NO",
	"AR87b_CAST006_WE",
	"AR87b_CAST007_WE",
	"AR87b_CAST008_CS1",
	"AR87b_CAST010_CS2",
	"AR87b_CAST011_CS3",
	"AR87b_CAST014_SO",
	"AR87b_CAST015_SO",
	"AR87b_CAST023_NO",
	"AR87b_CAST032_L10",
    "AR87b_CAST033_L8"
]
irregular_pairs = [x for x in pairs if x[0] in irregular_casts]

In [10]:
# Get summary statistics for casts w/ irregular sample intervals
optical_transmission_avg, total_volume_avg = summarize_cast_lisst(
    irregular_pairs
)

In [14]:
# load data for each of the casts (naming w/ leg letter and cast #)
datasets = list(())
loop = irregular_pairs.copy()
while len(loop)>0:
    key, value = loop.pop(0)
    lisst_ds = load_lisst((RDA_URL + value), csvhdr)
    lisst_ds = lisst_ds.assign_attrs(cast=key)
    # Mask data where depth < 0
    depth_mask = lisst_ds.time[lisst_ds.depth>=0].values
    lisst_ds = lisst_ds.sel(time=depth_mask)
    datasets.append(lisst_ds)
[a3, a4, a6, a7, a12, a14, b6, b7, b8, b10, b11, b14, b15, b23, b32, b33] = datasets

In [16]:
b6

<xarray.Dataset> Size: 199kB
Dimensions:                  (time: 400, bin: 36)
Coordinates:
  * time                     (time) datetime64[ns] 3kB 2025-04-12T20:44:08 .....
  * bin                      (bin) int64 288B 1 2 3 4 5 6 ... 31 32 33 34 35 36
Data variables: (12/26)
    laser_transmission       (time) float64 3kB 0.8787 0.8838 ... 1.015 1.004
    supply_voltage           (time) float64 3kB 10.69 10.69 ... 10.67 10.67
    ext_in1                  (time) float64 3kB -0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    laser_ref                (time) float64 3kB 0.923 0.922 ... 1.056 1.056
    depth                    (time) float64 3kB 0.45 2.27 1.84 ... 0.89 0.66
    temperature              (time) float64 3kB 9.697 11.41 11.83 ... 12.0 12.0
    ...                       ...
    rawpressure2             (time) int64 3kB 22605 22891 22824 ... 22675 22638
    ambient_light            (time) int64 3kB 10 7 14 12 8 11 ... 40 49 35 34 39
    ext_in3                  (time) float64 3kB 1.354 0.022 ... 1.834 0.025
    optical_transmission     (time) float64 3kB 0.984 0.991 ... 0.994 0.984
    beam_attenuation         (time) float64 3kB 0.641 0.37 0.752 ... 0.227 0.658
    volume_concentration_2D  (bin, time) float64 115kB 0.0 0.0 ... 0.0345 1.563
Attributes:
    cast:     AR87b_CAST006_WE

## Plot optical transmission against total volume concentration for casts

In [ ]:
# AR87a_CAST003_L10


In [ ]:
# AR87a_CAST004_GL564_GL376


In [ ]:
# AR87a_CAST006_SE


In [ ]:
# AR87a_CAST007_SE


In [ ]:
# AR87a_CAST012_AC1


In [ ]:
# AR87a_CAST014_NO


In [ ]:
# AR87b_CAST006_WE


In [ ]:
# AR87b_CAST007_WE


In [ ]:
# AR87b_CAST008_CS1
# AR87b_CAST010_CS2
# AR87b_CAST011_CS3
# AR87b_CAST014_SO
# AR87b_CAST015_SO
# AR87b_CAST023_NO
# AR87b_CAST032_L10
# AR87b_CAST033_L8